In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets, models
from tqdm import tqdm
import os

class ResumeTrainer:
    def __init__(self, 
                 model_path,
                 num_classes,
                 data_dir,
                 optimizer_type='AdamW',
                 learning_rate=1e-3,
                 weight_decay=1e-4,
                 start_epoch=0,
                 num_epochs=10,
                 batch_size=16,
                 device=None,
                 augmentations=None,
                 save_path=None,
                 scheduler=True,
                 early_stop_patience=8,
                 freeze_layers=True,          
                 layers_to_freeze=None):      

        self.model_path = model_path
        self.num_classes = num_classes
        self.data_dir = data_dir
        self.optimizer_type = optimizer_type
        self.lr = learning_rate
        self.weight_decay = weight_decay
        self.start_epoch = start_epoch
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        self.augmentations = augmentations or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        self.save_path = save_path or model_path
        self.scheduler_enabled = scheduler
        self.early_stop_patience = early_stop_patience
        self.freeze_layers = freeze_layers
        self.layers_to_freeze = layers_to_freeze

        # Placeholders
        self.model = None
        self.optimizer = None
        self.criterion = nn.CrossEntropyLoss()
        self.train_loader = None
        self.val_loader = None
        self.scheduler = None

        self.load_model()
        self.load_optimizer()
        self.prepare_data()


    def load_model(self):
        if os.path.exists(self.model_path):
            print(f"Loading checkpoint from {self.model_path}")
            checkpoint = torch.load(self.model_path, map_location=self.device)

            
            self.model = models.resnet18(weights=None)
            self.model.fc = nn.Linear(self.model.fc.in_features, self.num_classes)

            try:
                self.model.load_state_dict(checkpoint['model_state_dict'])
            except Exception as e:
                raise RuntimeError("Checkpoint is incompatible. Aborting.") from e

            self.start_epoch = checkpoint.get('epoch', self.start_epoch)
            print(f"Resuming from epoch {self.start_epoch}")

        else:
            print("No checkpoint found, initializing pretrained model")
            self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.model.fc = nn.Linear(self.model.fc.in_features, self.num_classes)

        # Freeze / unfreeze layers
        if self.freeze_layers:
            if self.layers_to_freeze:
                # Freeze only specified layers
                for name, param in self.model.named_parameters():
                    param.requires_grad = not any(name.startswith(layer) for layer in self.layers_to_freeze)
            else:
                # Freeze all backbone layers (everything except fc)
                for name, param in self.model.named_parameters():
                    if not name.startswith('fc'):
                        param.requires_grad = False

        self.model = self.model.to(self.device)


    def load_optimizer(self):
        if self.optimizer_type.lower() == 'adamw':
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay
            )
        elif self.optimizer_type.lower() == 'sgd':
            self.optimizer = torch.optim.SGD(
                self.model.parameters(),
                lr=self.lr,
                momentum=0.9
            )
        else:
            raise ValueError("Unsupported optimizer type")

        if os.path.exists(self.model_path):
            checkpoint = torch.load(self.model_path, map_location=self.device)
            opt_state = checkpoint.get('optimizer_state_dict', None)
            if opt_state is not None:
                self.optimizer.load_state_dict(opt_state)

        if self.scheduler_enabled:
            self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer,
                mode='min',
                factor=0.5,
                patience=3,
                min_lr=1e-6
            )


    def prepare_data(self):
        dataset = datasets.ImageFolder(
            root=self.data_dir,
            transform=self.augmentations
        )
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
        self.train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=self.batch_size)



    def train(self):
        best_val_acc = 0.0
        early_stop_counter = 0

        for epoch in range(self.start_epoch, self.start_epoch + self.num_epochs):
            self.model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            loop = tqdm(self.train_loader, desc=f"Epoch {epoch+1}", leave=False)
            for images, labels in loop:
                images, labels = images.to(self.device), labels.to(self.device)
                self.optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

                loop.set_postfix(loss=loss.item(), acc=correct/total*100)

            train_loss = running_loss / total
            train_acc = correct / total * 100

            val_loss, val_acc = self.validate()
            print(f"Epoch {epoch+1}: "
                  f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, "
                  f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

            if self.scheduler_enabled:
                self.scheduler.step(val_loss)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                early_stop_counter = 0
                self.save_checkpoint(epoch+1)
            else:
                early_stop_counter += 1
                if early_stop_counter >= self.early_stop_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break


    def validate(self):
        self.model.eval()
        correct = 0
        total = 0
        running_loss = 0.0
        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        return running_loss / total, correct / total * 100



    def save_checkpoint(self, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict()
        }
        torch.save(checkpoint, self.save_path)
        print(f"Saved checkpoint → {self.save_path}")



    def evaluate(self, test_loader):
        self.model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        acc = correct / total * 100
        print(f"Test Accuracy: {acc:.2f}%")
        return acc

In [ ]:
from torchvision import transforms
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

trainer = ResumeTrainer(
    model_path='/content/drive/MyDrive/ColabNotebooks/document-classifier/models/trained_models/best_resnet18.pth',
    data_dir='/content/drive/MyDrive/ColabNotebooks/document-classifier/data/scanned_docs',
    num_classes=10,
    optimizer_type='AdamW',
    batch_size=32,
    num_epochs=1,
    learning_rate=0.002,
    augmentations=train_transforms,
    freeze_layers=True,
    layers_to_freeze=['layer1', 'layer2']  
)

trainer.train()

Loading checkpoint from /content/drive/MyDrive/ColabNotebooks/document-classifier/models/trained_models/best_resnet18.pth


Resuming from epoch 8


Epoch 9: Train Loss=0.9240, Train Acc=70.56%, Val Loss=1.2577, Val Acc=59.54%
Saved checkpoint → /content/drive/MyDrive/ColabNotebooks/document-classifier/models/trained_models/best_resnet18.pth


Epoch 10: Train Loss=0.5544, Train Acc=82.01%, Val Loss=1.1182, Val Acc=69.30%
Saved checkpoint → /content/drive/MyDrive/ColabNotebooks/document-classifier/models/trained_models/best_resnet18.pth


Epoch 11: Train Loss=0.4391, Train Acc=85.53%, Val Loss=2.3765, Val Acc=46.92%


...

Epoch 14: Train Loss=0.2094, Train Acc=93.86%, Val Loss=1.1024, Val Acc=68.29%


Epoch 15: Train Loss=0.1806, Train Acc=93.61%, Val Loss=2.2882, Val Acc=46.20%


Epoch 16: Train Loss=0.1962, Train Acc=93.11%, Val Loss=0.8639, Val Acc=75.61%


Epoch 17: Train Loss=0.0822, Train Acc=97.49%, Val Loss=0.5001, Val Acc=85.94%
Saved checkpoint → /content/drive/MyDrive/ColabNotebooks/document-classifier/models/trained_models/best_resnet18.pth


Epoch 18: Train Loss=0.0349, Train Acc=99.10%, Val Loss=0.6911, Val Acc=83.64%
